In [1]:
"""FEMA disaster declarations (California): fetch -> save RAW -> clean -> save PROCESSED.

Source: OpenFEMA DisasterDeclarationsSummaries v2 (no API key needed).
Each row is one *county* (designated area) within a declaration, so a single
disaster usually produces many rows.

Run directly:  python fema_declarations.py
Or import:     from fema_declarations import run
"""
import json
import logging
import time
from pathlib import Path

In [2]:
import pandas as pd
import requests

In [3]:
logger = logging.getLogger(__name__)

In [4]:
BASE_URL = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"
STATE = "CA"
START_YEAR = 2020
# Matches our integration goal. Set to None to keep every incident type.
INCIDENT_TYPES = ["Earthquake", "Fire"]
PAGE_SIZE = 1000  # OpenFEMA's max records per request

In [5]:
RAW_DIR = Path("data/raw/fema")
PROCESSED_DIR = Path("data/processed")

In [6]:
KEEP_COLUMNS = [
    "id", "femaDeclarationString", "disasterNumber", "state", "declarationType",
    "declarationDate", "incidentType", "declarationTitle",
    "incidentBeginDate", "incidentEndDate",
    "ihProgramDeclared", "iaProgramDeclared", "paProgramDeclared", "hmProgramDeclared",
    "fipsStateCode", "fipsCountyCode", "designatedArea",
]

In [7]:
def fetch_page(skip: int, retries: int = 3) -> list[dict]:
    """Fetch one page of California declarations. Retries with exponential backoff."""
    params = {
        "$filter": f"state eq '{STATE}'",
        "$top": PAGE_SIZE,
        "$skip": skip,
        "$orderby": "id",  # stable order so paging doesn't skip or repeat rows
    }
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(BASE_URL, params=params, timeout=60)
            resp.raise_for_status()
            return resp.json()["DisasterDeclarationsSummaries"]
        except (requests.RequestException, KeyError, ValueError) as exc:
            logger.warning("FEMA skip=%d attempt %d/%d failed: %s", skip, attempt, retries, exc)
            if attempt == retries:
                logger.error("Giving up on FEMA skip=%d", skip)
                raise
            time.sleep(2 ** attempt)

In [8]:
def fetch_all() -> list[dict]:
    """Page through the API until a short page signals the end."""
    records, skip = [], 0
    while True:
        page = fetch_page(skip)
        records.extend(page)
        logger.info("Fetched %d records so far", len(records))
        if len(page) < PAGE_SIZE:
            break
        skip += PAGE_SIZE
    return records

In [9]:
def save_raw(records: list[dict]) -> Path:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    path = RAW_DIR / "fema_declarations_ca.json"
    path.write_text(json.dumps(records))
    logger.info("Saved raw -> %s (%d records)", path, len(records))
    return path

In [10]:
def clean(records: list[dict], start_year: int = START_YEAR) -> pd.DataFrame:
    df = pd.DataFrame(records)
    df = df[[c for c in KEEP_COLUMNS if c in df.columns]].copy()

    for col in ["declarationDate", "incidentBeginDate", "incidentEndDate"]:
        df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")

    df = df.drop_duplicates(subset="id")
    df = df[df["declarationDate"].dt.year >= start_year]
    if INCIDENT_TYPES:
        df = df[df["incidentType"].isin(INCIDENT_TYPES)]

    # 5-digit county FIPS (state 2 + county 3), e.g. "06037" = Los Angeles County.
    # County code "000" means a tribal area or statewide row, not a real county,
    # so we leave county_fips empty there to avoid bad joins with Census data.
    state = df["fipsStateCode"].astype(str).str.zfill(2)
    county = df["fipsCountyCode"].astype(str).str.zfill(3)
    df["county_fips"] = (state + county).where(county != "000")

    return df.sort_values("declarationDate").reset_index(drop=True)

In [11]:
def run(start_year: int = START_YEAR) -> pd.DataFrame:
    records = fetch_all()
    if not records:
        raise RuntimeError("No FEMA data collected")
    save_raw(records)

    df = clean(records, start_year)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    out = PROCESSED_DIR / "fema_declarations_ca_clean.csv"
    df.to_csv(out, index=False)
    logger.info("Saved processed -> %s (%d rows)", out, len(df))
    return df

In [12]:
if __name__ == "__main__":
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    )
    run()

2026-10-01 13:48:20,760 INFO __main__: Fetched 1000 records so far
2026-10-01 13:48:21,272 INFO __main__: Fetched 1691 records so far
2026-10-01 13:48:21,291 INFO __main__: Saved raw -> data/raw/fema/fema_declarations_ca.json (1691 records)
2026-10-01 13:48:21,357 INFO __main__: Saved processed -> data/processed/fema_declarations_ca_clean.csv (116 rows)


In [ ]:
records = fetch_all()                      # every CA record, no filters
raw = pd.DataFrame(records)

eq = raw[raw["incidentType"] == "Earthquake"]
print(len(eq), "earthquake rows in all of CA history")
print(eq[["declarationDate", "declarationTitle", "declarationType"]]
      .drop_duplicates().sort_values("declarationDate"))

2026-10-01 13:54:26,888 INFO __main__: Fetched 1000 records so far
2026-10-01 13:54:27,568 INFO __main__: Fetched 1691 records so far


32 earthquake rows in all of CA history
               declarationDate          declarationTitle declarationType
535   1971-02-09T00:00:00.000Z   SAN FERNANDO EARTHQUAKE              DR
1044  1979-10-19T00:00:00.000Z                EARTHQUAKE              DR
376   1983-05-05T00:00:00.000Z       COALINGA EARTHQUAKE              DR
718   1987-10-07T00:00:00.000Z  EARTHQUAKE & AFTERSHOCKS              DR
380   1989-10-18T00:00:00.000Z    LOMA PRIETA EARTHQUAKE              DR
551   1992-05-04T00:00:00.000Z  EARTHQUAKE & AFTERSHOCKS              DR
354   1992-07-02T00:00:00.000Z  EARTHQUAKE & AFTERSHOCKS              DR
145   1994-01-17T00:00:00.000Z     NORTHRIDGE EARTHQUAKE              DR
170   2000-09-14T00:00:00.000Z                EARTHQUAKE              DR
141   2004-01-13T00:00:00.000Z                EARTHQUAKE              DR
1512  2010-05-07T00:00:00.000Z                EARTHQUAKE              DR
72    2014-09-11T00:00:00.000Z                EARTHQUAKE              DR
177   2019-

In [19]:
fema = pd.read_csv(
    "data/processed/fema_declarations_ca_clean.csv",
    dtype={"county_fips": str, "fipsStateCode": str, "fipsCountyCode": str},
    parse_dates=["declarationDate"],
)
print(fema.shape)
print(fema["declarationDate"].min(), "to", fema["declarationDate"].max())
print(fema["declarationDate"].dt.year.value_counts().sort_index())

(217, 18)
2016-06-05 00:00:00+00:00 to 2026-08-04 00:00:00+00:00
declarationDate
2016    14
2017    45
2018    29
2019    13
2020    65
2021    21
2022     6
2023     1
2024    13
2025     6
2026     4
Name: count, dtype: int64


In [20]:
print(fema["state"].unique())                       # should be ['CA']
print(fema["incidentType"].value_counts())          # only Earthquake and Fire
print(fema["declarationType"].value_counts())       # mostly FM, some DR/EM

['CA']
incidentType
Fire          214
Earthquake      3
Name: count, dtype: int64
declarationType
FM    142
DR     60
EM     15
Name: count, dtype: int64


In [21]:
print(fema["id"].is_unique)                                  # True
print(fema["county_fips"].isna().sum(), "rows with no county FIPS")
print(fema["county_fips"].dropna().str.len().unique())       # [5]
print(fema["county_fips"].dropna().str[:2].unique())         # ['06']
print(fema["county_fips"].nunique(), "of 58 CA counties")

True
3 rows with no county FIPS
[5]
['06']
41 of 58 CA counties


In [22]:
fema[fema["county_fips"].isna()][["declarationTitle", "designatedArea"]].drop_duplicates()

,declarationTitle,designatedArea
71,MENDOCINO FIRE COMPLEX,Hopland Rancheria (Indian Reservation)
102,APPLE FIRE,Morongo Indian Reservation
193,EARTHQUAKE,Rohnerville Rancheria (Indian Reservation)


In [23]:
print((fema["incidentBeginDate"] > fema["declarationDate"]).sum())   # should be 0 or very small
fema.groupby("disasterNumber")["county_fips"].nunique().describe()   # counties per declaration

3


count    125.000000
mean       1.712000
std        2.199032
min        0.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       20.000000
Name: county_fips, dtype: float64

In [25]:
fema.groupby("incidentType")["disasterNumber"].nunique()

incidentType
Earthquake      2
Fire          123
Name: disasterNumber, dtype: int64